In [3]:
import pandas as pd
import seaborn as sns
import numpy as np 
import matplotlib.pyplot as plt

import re,string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences


nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aximsoft/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/aximsoft/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/aximsoft/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
df = pd.read_csv(r"../data/raw/IMDB Dataset.csv")

In [5]:
duplicate = df.duplicated()
df[duplicate]
df = df.drop_duplicates()

In [6]:
#lower case tranformation

In [7]:
df["cleaned_review"] = df["review"].str.lower()

In [8]:
df

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is..."
...,...,...,...
49995,I thought this movie did a down right good job...,positive,i thought this movie did a down right good job...
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,"bad plot, bad dialogue, bad acting, idiotic di..."
49997,I am a Catholic taught in parochial elementary...,negative,i am a catholic taught in parochial elementary...
49998,I'm going to have to disagree with the previou...,negative,i'm going to have to disagree with the previou...


In [9]:
df["sentiment"].unique()

<StringArray>
['positive', 'negative']
Length: 2, dtype: str

In [10]:
#remove html tags

In [11]:
df["cleaned_review"] = df["cleaned_review"].apply(
    lambda x: re.sub(r"<.*?>", "", x)
)

In [12]:
df

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. the filming tec...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is..."
...,...,...,...
49995,I thought this movie did a down right good job...,positive,i thought this movie did a down right good job...
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,"bad plot, bad dialogue, bad acting, idiotic di..."
49997,I am a Catholic taught in parochial elementary...,negative,i am a catholic taught in parochial elementary...
49998,I'm going to have to disagree with the previou...,negative,i'm going to have to disagree with the previou...


In [13]:
#punchuation and unwanted character

In [14]:
df["cleaned_review"] = df["cleaned_review"].apply(
    lambda x: x.translate(str.maketrans("", "", string.punctuation))
)

In [15]:
df["cleaned_review"] = df["cleaned_review"].str.replace(r"\s+", " ", regex=True).str.strip()

In [16]:
df

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love in the time of money is a ...
...,...,...,...
49995,I thought this movie did a down right good job...,positive,i thought this movie did a down right good job...
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,bad plot bad dialogue bad acting idiotic direc...
49997,I am a Catholic taught in parochial elementary...,negative,i am a catholic taught in parochial elementary...
49998,I'm going to have to disagree with the previou...,negative,im going to have to disagree with the previous...


In [17]:
df['review'].max()

'ý thýnk uzak ýs the one of the best films of all times and everybody must realize this movie.I m a Turkish boy and a big cinema fun. and in this days our cinema industry is highing up.And UZAK is the best Turkish film of last ten years.and maybe one of the best films of all times.director nuri bilge ceylan is quite amazing.telling story,characters,atmosphere is wonderful.he is a minimalist director and tells about routine event family,dreams,expects,life.tells about you ,tells about me,tells about us.I promise you will find a piece of your body in this movie.cinema life welcomes a new director.he is waiting to realize.I promise yo you will love this movie please watch it'

In [18]:
#stop words

In [19]:
stop_words = set(stopwords.words("english"))

In [20]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [21]:
[word for word in ["not", "no", "nor", "never"] if word in stop_words]

['not', 'no', 'nor']

In [22]:
stop_words.difference_update([    "no",
    "not",
    "nor",
    "never",
    "neither",
    "none",
    "nobody",
    "nothing",
    "nowhere",
    "hardly",
    "scarcely",
    "barely",
    "isn't",
    "wasn't",
    "didn't",
    "don't",
    "can't",
    "couldn't",
    "won't"])

In [23]:
df["cleaned_review"] = df["cleaned_review"].apply(
    lambda x: " ".join(word for word in x.split() if word not in stop_words)
)

In [24]:
df.head(10)

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching 1 oz episode ...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...
5,"Probably my all-time favorite movie, a story o...",positive,probably alltime favorite movie story selfless...
6,I sure would like to see a resurrection of a u...,positive,sure would like see resurrection dated seahunt...
7,"This show was an amazing, fresh & innovative i...",negative,show amazing fresh innovative idea 70s first a...
8,Encouraged by the positive comments about this...,negative,encouraged positive comments film looking forw...
9,If you like original gut wrenching laughter yo...,positive,like original gut wrenching laughter like movi...


In [25]:
#tonkenization

In [26]:
df["cleaned_review"] = df["cleaned_review"].apply(word_tokenize)

In [27]:
df

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,"[one, reviewers, mentioned, watching, 1, oz, e..."
1,A wonderful little production. <br /><br />The...,positive,"[wonderful, little, production, filming, techn..."
2,I thought this was a wonderful way to spend ti...,positive,"[thought, wonderful, way, spend, time, hot, su..."
3,Basically there's a family where a little boy ...,negative,"[basically, theres, family, little, boy, jake,..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"[petter, matteis, love, time, money, visually,..."
...,...,...,...
49995,I thought this movie did a down right good job...,positive,"[thought, movie, right, good, job, wasnt, crea..."
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,"[bad, plot, bad, dialogue, bad, acting, idioti..."
49997,I am a Catholic taught in parochial elementary...,negative,"[catholic, taught, parochial, elementary, scho..."
49998,I'm going to have to disagree with the previou...,negative,"[im, going, disagree, previous, comment, side,..."


In [28]:
#data setsplit

In [29]:
x= df["cleaned_review"]
y= df["sentiment"]

In [30]:
x_temp,x_test,y_temp,y_test = train_test_split(
    x,y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

x_train,x_val,y_train,y_val = train_test_split(
    x_temp,y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp)

In [31]:
print(x_test.shape)
print(y_test.shape)
print(x_val.shape)
print(y_val.shape)
print(x_train.shape)
print(y_train.shape)

(7438,)
(7438,)
(7439,)
(7439,)
(34705,)
(34705,)


In [32]:
#tokenization

In [33]:
tokenizer = Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)

In [34]:
tokenizer.fit_on_texts(x_train)

In [35]:
x_train_seq = tokenizer.texts_to_sequences(x_train)

x_val_seq = tokenizer.texts_to_sequences(x_val)

x_test_seq = tokenizer.texts_to_sequences(x_test)

In [36]:
print(x_train.iloc[0])
print(x_train_seq[0])

['not', 'much', 'say', 'one', 'except', 'probably', 'worst', 'early', 'spate', 'zombie', 'movies', 'may', 'get', 'watch', 'another', 'one', 'revolt', 'zombies', '1936', 'month', 'star', 'john', 'carradines', 'intention', 'building', 'army', 'service', 'third', 'reich', 'not', 'seen', 'much', 'james', 'baskett', 'uncle', 'remus', 'song', 'south', '1946', 'plays', 'leader', 'also', 'serves', 'carradines', 'manservant', 'black', 'comic', 'mantan', 'moreland', 'reprises', 'fraidy', 'cat', 'chauffeur', 'role', 'king', 'zombies', '1941', 'exotically', 'named', 'madame', 'sultewan', 'carradines', 'housekeeper', 'unfortunately', 'carradine', 'supreme', 'achievement', '\x96', 'zombification', 'wife', '\x96', 'brings', 'sorts', 'trouble', 'not', 'relatives', 'turn', 'remote', 'abodelab', 'inquire', 'sudden', 'death', 'means', 'fake', 'funeral', 'service', 'actually', 'proves', 'disobedient', 'indignant', 'eventually', 'persuading', 'fellow', 'zombies', 'rise', 'master', 'also', 'involved', 'cowb

In [37]:
#sequence

In [38]:
MAX_LEN = 200

In [39]:
x_train_pad = pad_sequences(
    x_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_val_pad = pad_sequences(
    x_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_test_pad = pad_sequences(
    x_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [40]:
print(x_train_pad.shape)
print(x_val_pad.shape)
print(x_test_pad.shape)

(34705, 200)
(7439, 200)
(7438, 200)


In [41]:
print(x_train_pad[0])

[    4    15    51     5   441   136   148   298     1   886    25   100
    17    32    71     5  8204  1114  7096  3394   235   208     1  3416
  1208  1106  2224   798 19150     4    34    15   485     1  1646     1
   466  1118  9123   181  1621    20  2329     1     1   220   621     1
     1 14654     1  1157 14241   119   625  1114  8205     1   644 12549
     1     1 10206   390  4761  5984  3431   373     1   225   373   812
  2438   961     4  4527   361  2759     1     1  1980   227   687  1017
  3745  2224    65  1433     1     1   730     1  1465  1114  2100  1033
    20   467  2770   235  1723  8448    50 19914   125  1755 10207    98
  1584  9123   181    79   929  1343  6146  3061  6146  2127  2424    88
     1     9    15    47  1963   255   416   659   160     1  9412     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0   

In [42]:
#taget vaariable

In [43]:
encoder = LabelEncoder()
y_train_en = encoder.fit_transform(y_train)
y_test_en = encoder.transform(y_test)
y_val_en  = encoder.transform(y_val)

In [44]:
print(y_train_en.shape)
print(y_test_en.shape)
print(y_val_en.shape)

(34705,)
(7438,)
(7439,)


In [45]:
np.save(r"../data/cleaned/x_train",x_train_pad)
np.save(r"../data/cleaned/x_test",x_test_pad)
np.save(r"../data/cleaned/x_val",x_val_pad)
np.save(r"../data/cleaned/y_train",y_train_en)
np.save(r"../data/cleaned/y_test",y_test_en)
np.save(r"../data/cleaned/y_val",y_val_en)

df.to_csv(r"../data/cleaned/cleaned_data.csv")